# ir_calendar_check_api_status — 從系統 API 確認資料真的進去了

- 用途：呼叫系統端唯一的讀取 API `GET <api_base_url>/ir-conferences/sync/status`，看 `app_company` / `app_ir_conference` 的筆數、最後同步時間、上一批 upsert 的 `created / updated / unchanged / failed` 與失敗明細；再跟 Databricks 這邊「我們送出去的東西」對照。
- **只讀**：不寫表、不寫 Volume、不動游標，也**不會 POST**（[c04] 會擋掉非 status 的路徑）。
- 輸入：系統 API status 端點、`b_{domain}_batch_log`、`s_{domain}_{company, conference}`、Volume `<volume_root>/ir_calendar/batches/<批次>/api/*.json`
- 輸出：stdout 的 PASS / FAIL / INFO 清單（[c20] 彙總），不產生任何表
- 參數（widgets）：`catalog`、`schema`、`domain`、`api_base_url`、`api_key_secret`、`status_path`、`verify_ssl`、`timeout_seconds`、`stale_hours`、`force_refresh`、`volume_root`、`api_fiscal_period`、`recent_batches`、`sample_rows`
- 排程：**不排程，人工執行**。真要排程請至少 1 小時一次，並只留 [c01]～[c11]
- 負責人 / 更新日期：（填）/ 2026-09-22

## 對 prod 的保護

| 做法 | 在哪 |
|---|---|
| 整份 notebook 只發 **1 次 GET**，回應快取在 `_STATUS_CACHE`；重跑後面的 cell 不會再打系統 | [c04] `fetch_status` |
| 要再打一次必須把 `force_refresh` 切成 `true`（預設 `false`） | [c01] / [c04] |
| 路徑必須以 `status` 結尾，寫入端點（`companies/sync`、`ir-conferences/sync`）直接 assert 擋掉 | [c01] / [c04] |
| 固定 `timeout_seconds`（預設 30 秒）、**不重試**：失敗就停，不製造 retry 風暴 | [c04] |
| 其餘檢查全在 Databricks 這邊（小表 count / limit、Volume 讀檔），不再碰系統 | [c12] / [c13] |

## 各 cell 在做什麼

| cell | 內容 |
|---|---|
| [c04] `io_api_status` | 唯一一次對外呼叫。API Key 只從 `dbutils.secrets` 取 |
| [c10] `show_status` | `data.tables[]`：每張系統表的 `rowCount`、`lastUpdatedAt`、距今幾小時。超過 `stale_hours`（規格 §5 的 24 小時）判 FAIL |
| [c11] `show_last_batch` | `lastCompaniesBatch` / `lastConferencesBatch`：上一批的統計與失敗明細（`failureSummaries`） |
| [c12] `compare_local` | Databricks 端對照：`batch_log.api_results` 最近幾批的 POST 結果 vs API 回報的上一批；silver 換算成系統 upsert key 的列數 vs `rowCount` |
| [c12a] `posted_totals` | 整張 `batch_log` 加總：歷次 POST 的 `rows_sent` / `created` / `updated` / `unchanged` / `failed`、批次數與起訖時間，再跟 `rowCount` 對照 |
| [c13] `show_sent_payload` | 從 Volume 歸檔的 `api/*.json` 印出我們實際送出的列（`sample_rows` 筆），肉眼比對值 |

## 讀這份報告要知道的事

- 「我塞了幾筆」有三個不同的數字：送出去的列數（[c12] 的 `rows`、[c13] 的 `rows=N`）、系統真的收下的（[c11] 的 `created / updated / unchanged`，只有上一批）、系統現在總共幾筆（[c10] 的 `rowCount`，含別的來源）。歷次累計看 [c12a]。
- envelope 把資料放在 `info`，不是定版規格寫的 `data`；`[c03] envelope_data` 兩種都收（`PAYLOAD_KEYS`），解析不出來時會把回應原文印出來。
- 系統回的時間字串**不帶時區，實測是台北時間**（規格 §6 寫 UTC）：`api_time_zone` 預設 `taipei`。設反了 `[c10]` 會標出「自算與 API 回報差約 8 小時」，切 widget 再跑 `[c10]` 即可（不會重打 API）。
- 系統端**沒有**逐列查詢的 API（規格 §8 只開 3 支，status 是唯一的讀取端點），所以「目前塞了哪些值」只能看到**表層統計 + 上一批結果**；列的內容要看 [c13]（我們送出去的原文）。
- `rowCount` 可能**大於**我們送的筆數：管理頁可手動新增公司 / 場次，軟刪除的列也可能仍算在內。因此 [c12] 只在 `rowCount < 本地期望值` 時判 FAIL。
- `api_fiscal_period` 預設 `as_is`（照爬蟲原值送），要填成跟 `ir_calendar_consume_batches` 那次執行**一樣的模式**，[c12] 的期望列數才對得上（系統的場次 upsert key 是 `(stock_code, fiscal_period)`）。
- `status_path` 預設用定版規格的 `ir-conferences/sync/status`；規格文末的原始版本寫的是 `ir/sync/status`，若系統端實作成後者，改 widget 即可。

Cell 標籤規則見 `docs/conventions.md`；分層設計見 `docs/20260921_ir_calendar_lakehouse_design.md`。


In [ ]:
# [c01] params
# 只讀：本 notebook 不寫表、不寫 Volume、不動游標，對系統 API 只發 GET。
# prod 保護：整份跑完只打 1 次 GET（[c04] 把回應快取起來）；要再打一次才把 force_refresh 切 true。
# catalog / schema / volume_root / api_fiscal_period 要跟 ir_calendar_consume_batches 那次執行一致，否則對照不到。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")
dbutils.widgets.text("volume_root", "/Volumes/micenter/mi3_datahub_prod/micenterfile_ext/unstructured_data_file")
dbutils.widgets.text("api_base_url", "")          # 到 /api/v1 為止，不含結尾斜線；空 = 不打 API，只看 Databricks 這邊
dbutils.widgets.text("api_key_secret", "")        # "scope/key"；憑證只走 dbutils.secrets
# 定版規格是 ir-conferences/sync/status；規格文末原始版本寫 ir/sync/status，系統端若用後者改這裡。
dbutils.widgets.text("status_path", "ir-conferences/sync/status")
dbutils.widgets.text("verify_ssl", "false")       # 內網自簽憑證
# 系統回的時間字串不帶時區。規格 §6 寫 UTC，但實測 lastUpdatedAt 與 hoursSinceLastUpdate 只有
# 當成台北時間才對得起來，所以做成 widget；系統端改掉就切回 utc。
dbutils.widgets.dropdown("api_time_zone", "taipei", ["taipei", "utc"])
dbutils.widgets.text("timeout_seconds", "30")     # 逾時就放棄，不重試
dbutils.widgets.text("stale_hours", "24")         # 超過幾小時沒更新視為異常（API 規格 §5）
dbutils.widgets.dropdown("force_refresh", "false", ["false", "true"])   # true = 允許本次執行再打一次 GET
# 與 consume [c01] 同名同值域：算「系統端 upsert key 的期望列數」時要用同一個模式，否則 [c12] 的對照沒意義。
FISCAL_PERIOD_MODES = ("quarter", "null", "as_is")
dbutils.widgets.dropdown("api_fiscal_period", "as_is", list(FISCAL_PERIOD_MODES))
dbutils.widgets.text("recent_batches", "10")      # batch_log 往回看幾批
dbutils.widgets.text("sample_rows", "10")         # [c13] 印幾列送出去的原文


def _flag(name: str) -> bool:
    return dbutils.widgets.get(name).strip().lower() == "true"


cfg = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "volume_root": dbutils.widgets.get("volume_root").strip().rstrip("/"),
    "api_base_url": dbutils.widgets.get("api_base_url").strip().rstrip("/"),
    "api_key_secret": dbutils.widgets.get("api_key_secret").strip(),
    "status_path": dbutils.widgets.get("status_path").strip().strip("/"),
    "api_fiscal_period": dbutils.widgets.get("api_fiscal_period").strip(),
    "api_time_zone": dbutils.widgets.get("api_time_zone").strip(),
    "verify_ssl": _flag("verify_ssl"),
    "force_refresh": _flag("force_refresh"),
    "timeout_seconds": float(dbutils.widgets.get("timeout_seconds")),
    "stale_hours": float(dbutils.widgets.get("stale_hours")),
    "recent_batches": int(dbutils.widgets.get("recent_batches")),
    "sample_rows": int(dbutils.widgets.get("sample_rows")),
}
assert cfg["catalog"] and cfg["schema"] and cfg["domain"], "catalog / schema / domain 不可為空"
assert cfg["volume_root"].startswith("/Volumes/"), "volume_root 必須是 UC Volume 路徑"
assert cfg["api_fiscal_period"] in FISCAL_PERIOD_MODES, f"api_fiscal_period 只能是 {FISCAL_PERIOD_MODES}"
# 只允許讀取端點：路徑必須以 status 結尾，且不能是兩支寫入端點（打錯就會對 prod 寫資料）。
assert cfg["status_path"].endswith("status"), f"status_path 必須以 status 結尾，實際是 {cfg['status_path']!r}"
assert 0 < cfg["timeout_seconds"] <= 120, "timeout_seconds 請設在 0～120 秒之間"
assert cfg["api_time_zone"] in ("taipei", "utc"), "api_time_zone 只能是 taipei / utc"
print({k: v for k, v in cfg.items() if k != "api_key_secret"})


In [ ]:
# [c02] imports
# 常數與 ir_calendar_consume_batches 的 [c02] / [c04] 對齊：那邊改了 LAYER_PREFIX、CONTROL_DIR、期別模式，這裡要一起改。
# 本 cell 與 [c03] 不碰 spark / dbutils / 網路，可被 tests/ 載入。
import json
import os
import re
from datetime import datetime, timedelta, timezone

from pyspark.sql import functions as F

UTC = timezone.utc  # noqa: UP017 - 本機測試仍是 Python 3.10，不用 datetime.UTC
TAIPEI = timezone(timedelta(hours=8))

LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}
CONTROL_DIR = "ir_calendar"          # <volume_root>/ir_calendar/：游標與批次歸檔

# 系統表 → 我們這邊的對照來源（silver 短名, 送這張表的寫入端點）。status 回應應該就是這兩張。
SYSTEM_TABLES = {
    "app_company": ("company", "companies/sync"),
    "app_ir_conference": ("conference", "ir-conferences/sync"),
}
# status 回應裡上一批結果的欄位 → 對應的寫入端點
LAST_BATCH_KEYS = {
    "lastCompaniesBatch": "companies/sync",
    "lastConferencesBatch": "ir-conferences/sync",
}
WRITE_ENDPOINTS = tuple(LAST_BATCH_KEYS.values())   # 本 notebook 絕不呼叫這兩支
BATCH_STAT_FIELDS = ("created", "updated", "unchanged", "failed")
# envelope 裡放資料的鍵：定版規格寫 data，實際系統回的是 info。依序找，找到第一個 dict 就用。
PAYLOAD_KEYS = ("data", "info", "result", "payload")
# 不帶時區的時間字串要當成哪一區（[c01] api_time_zone）。實測系統回的是台北時間，規格 §6 寫的是 UTC。
NAIVE_TZ_BY_NAME = {"taipei": TAIPEI, "utc": UTC}


In [ ]:
# [c03] status_helpers
# 純函式：解析 status 回應、時間換算、期別換算，外加檢查結果收集器。無 I/O、無網路。
# 每項檢查只記錄不 raise，讓全部跑完再由 [c20] 一次看。
CHECKS: list[tuple[str, bool | None, str]] = []


def check(name: str, ok: bool, detail: str = "") -> bool:
    CHECKS.append((name, bool(ok), detail))
    print(f"  {'PASS' if ok else 'FAIL'}  {name}" + (f" — {detail}" if detail else ""))
    return bool(ok)


def note(name: str, detail: str) -> None:
    """不判定對錯，只記錄觀察值。"""
    CHECKS.append((name, None, detail))
    print(f"  INFO  {name} — {detail}")


def table_name(layer: str, short: str) -> str:
    return f"{cfg['catalog']}.{cfg['schema']}.{LAYER_PREFIX[layer]}{cfg['domain']}_{short}"


_PAYLOAD_KEY_SEEN: set[str] = set()


def envelope_data(body: dict) -> dict:
    """系統 envelope（success / code / message / correlationId）→ 裝資料的那個物件。

    定版規格寫 data，實際系統回的是 info，所以 PAYLOAD_KEYS 依序找第一個 dict。
    envelope 說失敗、或找不到資料物件就拋（連原文一起印出來，不猜欄位）。
    """
    if not isinstance(body, dict):
        raise ValueError(f"回應不是 JSON 物件：{type(body).__name__}")
    if body.get("success") is False:
        raise RuntimeError(f"API 回 success=false：code={body.get('code')} message={body.get('message')}")
    for key in PAYLOAD_KEYS:
        v = body.get(key)
        if isinstance(v, dict):
            if key != PAYLOAD_KEYS[0] and key not in _PAYLOAD_KEY_SEEN:
                _PAYLOAD_KEY_SEEN.add(key)
                print(f"  注意：資料放在 {key!r}，不是規格寫的 {PAYLOAD_KEYS[0]!r}；照 {key!r} 解析")
            return v
    if isinstance(body.get("tables"), list):        # 沒有外層 envelope，整包就是資料
        return body
    raise ValueError(f"回應找不到資料物件（試過 {PAYLOAD_KEYS}）：code={body.get('code')} "
                     f"message={body.get('message')} 原文={json.dumps(body, ensure_ascii=False)[:500]}")


def parse_ts(s: str | None, naive_tz: timezone = UTC) -> datetime | None:
    """API 的 ISO 時間字串 → UTC datetime。不帶時區的字串視為 naive_tz（[c01] api_time_zone 決定）。

    規格 §6 說都是 UTC，但實測系統回的是台北時間，所以這裡不寫死。解析不了回 None，不猜。
    """
    if not s:
        return None
    try:
        ts = datetime.fromisoformat(str(s).replace("Z", "+00:00"))
    except ValueError:
        return None
    return ts.replace(tzinfo=naive_tz).astimezone(UTC) if ts.tzinfo is None else ts.astimezone(UTC)


def hours_since(s: str | None, now: datetime, naive_tz: timezone = UTC) -> float | None:
    """自己算距今幾小時，不完全依賴 API 的 hoursSinceLastUpdate（兩者差太多時 [c10] 會標出來）。"""
    ts = parse_ts(s, naive_tz)
    return None if ts is None else (now - ts).total_seconds() / 3600.0


def status_tables(data: dict) -> list[dict]:
    """data.tables[] → 正規化後的 list，缺欄位補 None。"""
    out = []
    for t in data.get("tables") or []:
        out.append({
            "table": t.get("tableName"),
            "rows": t.get("rowCount"),
            "last_updated_at": t.get("lastUpdatedAt"),
            "api_hours": t.get("hoursSinceLastUpdate"),
        })
    return out


def batch_stats(b: dict | None) -> dict:
    """lastCompaniesBatch / lastConferencesBatch → 統一鍵名；None 時回空 dict。"""
    if not isinstance(b, dict):
        return {}
    out = {"executed_at": b.get("executedAt"), "success": b.get("success"),
           "failures": list(b.get("failureSummaries") or [])}
    out.update({f: b.get(f) for f in BATCH_STAT_FIELDS})
    return out


def fmt_stats(s: dict) -> str:
    return " ".join(f"{f}={s.get(f)}" for f in BATCH_STAT_FIELDS)


def fiscal_period_for_api(value: str | None, mode: str) -> str | None:
    """= consume [c03] 的同名函式（那邊改了這裡要一起改）。只用來顯示 [c13] 實際送出的值。

    quarter：只留季別（2026Q3 / FY2026Q3 / Q3 → Q3），取不到就 None；null：一律 None；as_is：原值。
    """
    if mode == "as_is":
        return value
    if mode == "null":
        return None
    if mode != "quarter":
        raise ValueError(f"不認得的 api_fiscal_period：{mode}")
    m = re.search(r"Q([1-4])", str(value), re.IGNORECASE) if value is not None else None
    return f"Q{m.group(1)}" if m else None


def api_period_col(c, mode: str):
    """同一套規則的 Column 版：silver 的 fiscal_period → 送給系統的值，用來算 upsert key 的期望列數。"""
    if mode == "as_is":
        return c
    if mode == "null":
        return F.lit(None).cast("string")
    return F.nullif(F.concat(F.lit("Q"), F.regexp_extract(c, r"[Qq]([1-4])", 1)), F.lit("Q"))


In [ ]:
# [c04] io_api_status
# 唯一一次對外呼叫：GET <api_base_url>/<status_path>。read-only，不帶 body、不重試、逾時就放棄。
# 回應快取在 _STATUS_CACHE：後面的 cell 重跑幾次都不會再打 prod；要重取把 force_refresh 切 true 再跑本 cell。
import time

import requests
import urllib3

_STATUS_CACHE: dict = {}
API_TZ = NAIVE_TZ_BY_NAME[cfg["api_time_zone"]]   # 系統回的無時區時間字串當成哪一區


def api_key(cfg: dict) -> str | None:
    ref = cfg.get("api_key_secret") or ""
    if "/" not in ref:
        return None
    scope, key = ref.split("/", 1)
    return dbutils.secrets.get(scope=scope, key=key)


def fetch_status(cfg: dict) -> dict | None:
    """回系統 API 的 envelope；api_base_url 未填回 None（只做 Databricks 端檢查）。"""
    base = cfg["api_base_url"]
    if not base:
        print("  api_base_url 未填：不呼叫系統 API，只看 Databricks 這邊（[c12] / [c13]）")
        return None
    if "body" in _STATUS_CACHE and not cfg["force_refresh"]:
        print("  用本次執行已取得的回應（要重打 prod 請把 force_refresh 設 true）")
        return _STATUS_CACHE["body"]

    url = f"{base}/{cfg['status_path']}"
    # 再擋一次：寫入端點永遠不會從這裡送出去（[c01] 的 assert 之外的第二道）。
    assert cfg["status_path"].endswith("status") and not cfg["status_path"].endswith(WRITE_ENDPOINTS), (
        f"只允許呼叫 status 端點，擋下：{url}")

    if not cfg["verify_ssl"]:
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    headers = {"Accept": "application/json"}
    key = api_key(cfg)
    if key:
        headers["X-Api-Key"] = key
    else:
        print("  注意：api_key_secret 未填，沒帶 X-Api-Key，系統端可能回 401")

    t0 = time.monotonic()
    r = requests.get(url, headers=headers, timeout=cfg["timeout_seconds"], verify=cfg["verify_ssl"])
    elapsed = time.monotonic() - t0
    print(f"  GET {url} → HTTP {r.status_code}（{elapsed:.2f}s）")
    r.raise_for_status()          # 失敗就停在這裡，不重試
    body = r.json() if r.content else {}
    _STATUS_CACHE["body"] = body
    _STATUS_CACHE["fetched_at"] = datetime.now(UTC)
    return body


STATUS = fetch_status(cfg)
NOW = _STATUS_CACHE.get("fetched_at") or datetime.now(UTC)
print(json.dumps(STATUS, ensure_ascii=False, indent=1)[:2000] if STATUS else "（無 API 回應）")


In [ ]:
# [c10] show_status
# data.tables[]：系統各表目前幾筆、最後同步時間、距今幾小時。規格 §5：「超過 24 小時未更新」是系統端唯一的異常訊號。
# 距今小時數自己用 lastUpdatedAt 算一份，跟 API 回報的對不上時標 INFO（多半是兩邊對時區的認定不同）。
SYS_TABLES: dict[str, dict] = {}

if STATUS is None:
    note("系統 API status", "api_base_url 未填，略過 API 檢查，只看 Databricks 端")
else:
    data = envelope_data(STATUS)
    rows = status_tables(data)
    check("status 回應帶 tables[]", bool(rows), f"{len(rows)} 張表")
    header = f"{'table':<22}{'rows':>8}  {'lastUpdatedAt':<26}{'距今(自算)':>12}{'API 回報':>10}"
    print("\n" + header)
    for t in rows:
        SYS_TABLES[t["table"]] = t
        t["hours"] = hours_since(t["last_updated_at"], NOW, API_TZ)
        self_h = "-" if t["hours"] is None else f"{t['hours']:.1f}h"
        api_h = "-" if t["api_hours"] is None else f"{float(t['api_hours']):.1f}h"
        print(f"{str(t['table']):<22}{str(t['rows']):>8}  {str(t['last_updated_at']):<26}{self_h:>12}{api_h:>10}")
    print()

    for name in SYSTEM_TABLES:
        t = SYS_TABLES.get(name)
        if t is None:
            check(f"{name} 出現在 status", False, "回應沒有這張表：系統端還沒寫入過，或表名與規格不同")
            continue
        h, n_rows = t["hours"], t["rows"]
        detail = f"rows={n_rows}, 最後更新 {t['last_updated_at']}"
        detail += "（算不出距今時間）" if h is None else f"（{h:.1f} 小時前，門檻 {cfg['stale_hours']:.0f}）"
        check(f"{name} 更新時效", h is not None and h <= cfg["stale_hours"], detail)
        check(f"{name} 有資料", isinstance(n_rows, int) and n_rows > 0, f"rowCount={n_rows}")
        if h is not None and t["api_hours"] is not None and abs(float(t["api_hours"]) - h) > 1.0:
            note(f"{name} 時間基準",
                 f"API 回 {float(t['api_hours']):.1f}h、自算 {h:.1f}h（api_time_zone={cfg['api_time_zone']}）："
                 "差約 8 小時就是時區設反了，把 api_time_zone 切成另一個值再跑本 cell（不會重打 API）")


In [ ]:
# [c11] show_last_batch
# lastCompaniesBatch / lastConferencesBatch：系統記得的「上一批 upsert」結果與失敗明細。
# 這是規格裡唯一能看到逐列失敗原因的地方（例：查無對應公司 = 場次先於公司送，或 stock_code 對不到）。
LAST_BATCH: dict[str, dict] = {}     # 寫入端點 -> 統計

if STATUS is None:
    note("上一批 upsert 結果", "api_base_url 未填，略過")
else:
    data = envelope_data(STATUS)
    for field, endpoint in LAST_BATCH_KEYS.items():
        s = batch_stats(data.get(field))
        if not s:
            note(f"{field}", "回應沒有這個區塊：系統端還沒收過這支的批次")
            continue
        LAST_BATCH[endpoint] = s
        h = hours_since(s["executed_at"], NOW, API_TZ)
        age = "" if h is None else f"（{h:.1f} 小時前）"
        print(f"\n{field}  executedAt={s['executed_at']}{age}  success={s['success']}")
        print(f"  {fmt_stats(s)}")
        check(f"{field} success", s["success"] is True, f"success={s['success']}")
        check(f"{field} 無失敗列", (s.get("failed") or 0) == 0, f"failed={s.get('failed')}")
        if all((s.get(f) or 0) == 0 for f in BATCH_STAT_FIELDS):
            note(f"{field} 空批次",
                 "created / updated / unchanged / failed 全是 0：這批沒有處理任何列（送了空的 rows[]），"
                 "表示資料是更早的批次進去的，對照 [c12] 的 rows 欄看我們那次送了幾列")
        for line in s["failures"][:20]:
            print(f"    失敗：{line}")
        if len(s["failures"]) > 20:
            print(f"    …另有 {len(s['failures']) - 20} 筆失敗明細未列出")


In [ ]:
# [c12] compare_local
# Databricks 端對照，不再碰系統 API：
#   (1) batch_log.api_results：我們最近幾批實際 POST 出去的結果，跟 [c11] 系統記得的上一批比對
#   (2) silver 換算成系統 upsert key 的期望列數，跟 status 的 rowCount 比對
# 都是小表（公司 176 筆、場次數千筆），count / distinct 可接受；silver 只讀需要的欄位。
bl = spark.table(table_name("bronze", "batch_log"))
recent = (bl.orderBy(F.col("seq").desc()).limit(cfg["recent_batches"])
            .select("batch_id", "seq", "status", F.explode_outer("api_results").alias("a")))
LOCAL_POSTS: dict[str, dict] = {}      # 寫入端點 -> 我們最後一次 POST 的結果

print(f"{'batch_id':<20}{'endpoint':<22}{'rows':>6}{'http':>6}{'ok':>7}"
      f"{'created':>9}{'updated':>9}{'unchg':>7}{'failed':>8}  posted_at")
for r in recent.collect():          # 最多 recent_batches × 端點數 列
    a = r["a"]
    if a is None:
        print(f"{r['batch_id']:<20}{'(這批沒有 POST)':<22}")
        continue
    print(f"{r['batch_id']:<20}{str(a['endpoint']):<22}{str(a['rows']):>6}{str(a['http_status']):>6}"
          f"{str(a['success']):>7}{str(a['created']):>9}{str(a['updated']):>9}{str(a['unchanged']):>7}"
          f"{str(a['failed']):>8}  {a['posted_at']}")
    prev = LOCAL_POSTS.get(a["endpoint"])
    if a["posted_at"] and (prev is None or (prev["posted_at"] and a["posted_at"] > prev["posted_at"])):
        LOCAL_POSTS[a["endpoint"]] = {"batch_id": r["batch_id"], **a.asDict()}
print()

# (1) 我們最後一次 POST vs 系統記得的上一批：對得起來，才確定系統收到的就是這一批。
for endpoint, local in LOCAL_POSTS.items():
    remote = LAST_BATCH.get(endpoint)
    if not remote:
        note(f"{endpoint} 批次對照", f"本地最後 POST 於 {local['posted_at']}（batch {local['batch_id']}），"
                                     "但 status 沒有對應區塊可比")
        continue
    same = all(local.get(f) == remote.get(f) for f in BATCH_STAT_FIELDS)
    check(f"{endpoint} 上一批統計一致", same,
          f"本地 {fmt_stats(local)}（batch {local['batch_id']}）／系統 {fmt_stats(remote)}"
          + ("" if same else " ← 系統記得的不是我們最後那批，或中間有別的來源寫入"))

# (2) 期望列數 vs rowCount。系統表可能另有管理頁手動新增、或軟刪除仍計入的列，
#     所以只在 rowCount 少於本地期望值時判 FAIL（= 有資料沒進去）。
mode = cfg["api_fiscal_period"]
comp = spark.table(table_name("silver", "company")).select("stock_code")
code_ok = comp["stock_code"].isNotNull() & (F.trim(comp["stock_code"]) != "")
n_comp_total, n_comp_code = comp.count(), comp.filter(code_ok).select("stock_code").distinct().count()
n_comp_nocode = comp.filter(~code_ok).count()
note("silver company", f"{n_comp_total} 列，其中 {n_comp_code} 個不重複 stock_code")
if n_comp_nocode:
    note("company 缺 stock_code",
         f"{n_comp_nocode} 列沒有 stock_code。規格 request 範例寫「可為 null（未上市公司）」，"
         "§3 失敗條件又寫「缺值 → 整批退回」——要跟系統端確認；真的整批退回會在 [c11] 看到 success=false")

conf = spark.table(table_name("silver", "conference")).select("stock_code", "fiscal_period")
conf_code_ok = conf["stock_code"].isNotNull() & (F.trim(conf["stock_code"]) != "")
# fp = 實際會送出去的 fiscalPeriod；fiscal_period 原值要留著，最後那段塌陷檢查要用。
sent = conf.filter(conf_code_ok).select(
    "stock_code", "fiscal_period", api_period_col(F.col("fiscal_period"), mode).alias("fp"))
n_conf_key = sent.select("stock_code", "fp").distinct().count()
n_conf_nocode, n_conf_total = conf.filter(~conf_code_ok).count(), conf.count()
note("silver conference", f"{n_conf_total} 列 → 送出去會落在 {n_conf_key} 個 (stock_code, fiscalPeriod) 鍵"
                          f"（api_fiscal_period={mode}）")
if n_conf_nocode:
    note("conference 缺 stock_code", f"{n_conf_nocode} 列；這些列在系統端會逐列失敗（不自動建公司，規格 §4）")

EXPECTED_ROWS = {"app_company": n_comp_code, "app_ir_conference": n_conf_key}
for name, expected in EXPECTED_ROWS.items():
    t = SYS_TABLES.get(name)
    if not t or not isinstance(t["rows"], int):
        note(f"{name} 筆數對照", f"沒有 status 可比，本地期望至少 {expected} 列")
        continue
    check(f"{name} 筆數不少於本地期望", t["rows"] >= expected,
          f"系統 {t['rows']} 列 vs 本地期望 {expected} 列"
          + ("（多出來的可能是管理頁手動新增或軟刪除仍計入）" if t["rows"] > expected else ""))

# fiscalPeriod 是必填，送出去是 null 的列在系統端會逐列失敗。三種模式都要檢查：
# as_is 是 fiscal_period 本身空的，quarter 是換算不出季別（FULL_YEAR 等），null 模式則是全部。
n_fp_null = sent.filter(F.col("fp").isNull()).count()
if n_fp_null:
    note("fiscalPeriod 會送 null",
         f"{n_fp_null} 列送出去的 fiscalPeriod 是 null（api_fiscal_period={mode}）；"
         "該欄必填，這些列在系統端會逐列失敗")

# 只有 quarter / null 會把不同年度的同一季塌成同一個 upsert key，系統端同鍵只留最後送進去的那筆。
if mode != "as_is":
    collide = (sent.groupBy("stock_code", "fp").agg(F.countDistinct("fiscal_period").alias("n"))
                   .filter(F.col("n") > 1))
    n_collide = collide.count()
    if n_collide:
        note("期別塌陷", f"{n_collide} 個鍵由多個 fiscal_period 值塌成同一個（api_fiscal_period={mode}）："
                         "系統端同一鍵只會留最後送進去的那筆")
        collide.limit(5).show(truncate=False)


In [ ]:
# [c12a] posted_totals
# 「我到底塞了幾筆」：把整張 batch_log 的 api_results 攤平加總（batch_log 每批一列，小表，可以全算）。
# 三個數字意義不同，別混用：
#   rows_sent = 我們送出去的列數（每批都會再送一次沒變的場次，會重複計，不等於資料筆數）
#   created   = 系統端真的新增的列數 ← 最接近「我塞了幾筆進去」
#   updated / unchanged = 該列已存在，內容有變 / 沒變
bl_all = (spark.table(table_name("bronze", "batch_log"))
          .select("batch_id", F.explode("api_results").alias("a"))
          .select("batch_id", "a.*")
          .filter(F.col("http_status").isNotNull()))   # 只算真的送出去的：dry_run / 沒填 api_base_url 不算

totals = (bl_all.groupBy("endpoint")
          .agg(F.countDistinct("batch_id").alias("batches"),
               F.sum("rows").alias("rows_sent"),
               F.sum("created").alias("created"),
               F.sum("updated").alias("updated"),
               F.sum("unchanged").alias("unchanged"),
               F.sum("failed").alias("failed"),
               F.min("posted_at").alias("first_post"),
               F.max("posted_at").alias("last_post"))
          .orderBy("endpoint"))
totals.show(truncate=False)

# 累計 created vs 系統現在的 rowCount：created 是我們建立過的列數，系統列數少於它代表中間有列被刪掉或沒進去。
CREATED_BY_ENDPOINT = {r["endpoint"]: (r["created"] or 0) for r in totals.collect()}
for name, (_short, endpoint) in SYSTEM_TABLES.items():
    created = CREATED_BY_ENDPOINT.get(endpoint)
    if created is None:
        note(f"{name} 累計 created", f"batch_log 沒有 {endpoint} 的成功 POST 紀錄")
        continue
    t = SYS_TABLES.get(name)
    detail = f"我們歷次 POST 累計 created={created}"
    if t and isinstance(t["rows"], int):
        detail += f"，系統現在 rowCount={t['rows']}"
        check(f"{name} 列數不少於累計 created", t["rows"] >= created, detail)
    else:
        note(f"{name} 累計 created", detail + "（沒有 status 可比）")


In [ ]:
# [c13] show_sent_payload
# 「到底塞了哪些值」：系統端沒有逐列查詢的 API，所以看我們送出去的原文——
# Volume 歸檔的 <volume_root>/ir_calendar/batches/<批次>/api/*.json。純讀檔，不碰系統。
# 場次列另外印出 fiscalPeriod 的原值與實際送出的值（consume 送出前會依 api_fiscal_period 換算）。
# 對象：最後一次真的 POST 出去的那批（[c12] 已從 batch_log 整理好）；都沒 POST 過就看最新一批。
if LOCAL_POSTS:
    target = max(LOCAL_POSTS.values(), key=lambda p: p["posted_at"])["batch_id"]
else:
    latest = bl.orderBy(F.col("seq").desc()).limit(1).collect()
    target = latest[0]["batch_id"] if latest else None

api_dir = f"{cfg['volume_root']}/{CONTROL_DIR}/batches/{target}/api" if target else None
if not api_dir or not os.path.isdir(api_dir):
    note("送出內容", f"找不到歸檔目錄 {api_dir}：這批沒有 api/*.json，或 Volume 路徑不同")
else:
    print(f"批次 {target} 送出的 API body（{api_dir}）\n")
    files = sorted(f for f in os.listdir(api_dir) if f.endswith(".json"))
    check("歸檔有 api/*.json", bool(files), f"{len(files)} 個檔")
    for fn in files:
        with open(os.path.join(api_dir, fn), encoding="utf-8") as fh:
            body = json.load(fh)
        rows = body.get("rows") or []
        print(f"  {fn}: generatedAt={body.get('generatedAt')} rows={len(rows)}")
        for row in rows[:cfg["sample_rows"]]:       # 只印前 sample_rows 列，不倒整份出來
            line = json.dumps(row, ensure_ascii=False)
            if "fiscalPeriod" in row:
                sent_fp = fiscal_period_for_api(row.get("fiscalPeriod"), cfg["api_fiscal_period"])
                line = f"fiscalPeriod {row.get('fiscalPeriod')!r} → 送出 {sent_fp!r} | " + line
            print(f"    {line[:400]}")
        if len(rows) > cfg["sample_rows"]:
            print(f"    …另有 {len(rows) - cfg['sample_rows']} 列未列出")
        print()


In [ ]:
# [c20] summary
# 一次看完。這份報告能回答的是「系統表有多少筆、最後一批進去的結果」與「我們送了什麼」；
# 系統端沒有逐列查詢 API，要逐列核對只能請系統端直接查 DB。
n_pass = sum(1 for _, ok, _ in CHECKS if ok is True)
n_fail = sum(1 for _, ok, _ in CHECKS if ok is False)
print(f"檢查 {n_pass + n_fail} 項，PASS {n_pass} / FAIL {n_fail}"
      f"（API 呼叫次數：{1 if _STATUS_CACHE else 0}）\n")
for name, ok, detail in CHECKS:
    mark = {True: "PASS", False: "FAIL", None: "INFO"}[ok]
    print(f"{mark}  {name}" + (f" — {detail}" if detail else ""))

if n_fail:
    print("\n有沒過的項目，常見原因：")
    print("  - 更新時效 FAIL：consume job 沒跑、或 api_base_url 留空跑的（資料只進 Databricks 沒送系統）")
    print("  - 上一批統計對不上：系統記得的不是我們最後那批，先看 [c12] 的 posted_at 與 executedAt")
    print("  - 筆數少於期望：看 [c11] 的 failureSummaries，多半是『查無對應公司』（公司主檔要先送）")
else:
    print("\n系統端與 Databricks 端對得上。要逐列核對值，把 [c13] 印出的 body 交給系統端比對 DB。")
